Spark Structured Streaming

In [1]:
//Inicialización de la sesión

// CELDA T0 — Inicialización para los ejemplos de teoría
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import org.apache.spark.sql.streaming.Trigger
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File

val spark = SparkSession.builder()
  .appName("Dia18S1_Teoria")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "2")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")

// Schema de eventos — se reutiliza en todos los ejemplos de teoría y práctica
val schemaEventos = StructType(Array(
  StructField("timestamp", StringType, nullable = true),
  StructField("usuario",   StringType, nullable = true),
  StructField("accion",    StringType, nullable = true),
  StructField("ciudad",    StringType, nullable = true)
))

// Utilidad para borrar carpeta completa — necesaria para limpiar checkpoints
def borrarCarpeta(ruta: String): Unit = {
  val f = new File(ruta)
  if (f.exists()) {
    f.listFiles().foreach { sub =>
      if (sub.isDirectory) borrarCarpeta(sub.getAbsolutePath)
      else sub.delete()
    }
    f.delete()
  }
}

// Crear carpetas y fichero de datos para los ejemplos
val carpetaInput = "C:/Curso-Scala/stream-input"
val carpetaCheck = "C:/Curso-Scala/stream-checkpoint"

Seq(carpetaInput, carpetaCheck).foreach { c =>
  borrarCarpeta(c)
  Files.createDirectories(Paths.get(c))
}

val datosTeo =
  """timestamp,usuario,accion,ciudad
    |2024-05-01 10:00:01,user001,login,Madrid
    |2024-05-01 10:00:03,user002,compra,Barcelona
    |2024-05-01 10:00:05,user001,busqueda,Madrid
    |2024-05-01 10:00:07,user003,login,Sevilla
    |2024-05-01 10:00:09,user002,pago,Barcelona""".stripMargin

Files.write(
  Paths.get(s"$carpetaInput/eventos_teoria.csv"),
  datosTeo.getBytes(StandardCharsets.UTF_8)
)

println(s"✅ Spark ${spark.version} listo — Scala ${scala.util.Properties.versionString}")
println("✅ schemaEventos, borrarCarpeta y datos de teoría listos")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 12:12:37 INFO SparkContext: Running Spark version 4.1.1
26/05/06 12:12:37 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/06 12:12:37 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/05/06 12:12:37 INFO ResourceUtils: ==============================================================
26/05/06 12:12:37 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/06 12:12:37 INFO ResourceUtils: ==============================================================
26/05/06 12:12:37 INFO SparkContext: Submitted application: Dia18S1_Teoria
26/05/06 12:12:37 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/06 12:12:37 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/06 12:12:37 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/06 12:12:37 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/05/06 12:12:37 INFO SecurityManager: Changing view acl

2026-05-06T10:12:38.007432500Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import org.apache.spark.sql.streaming.Trigger
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@53959cef
import spark.implicits._
schemaEventos: StructType = Seq(
  StructField(
    name = "timestamp",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "usuario",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "accion",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "ciudad",
    dataType = StringType,
    nullable = true,
    metadata = {}
  )
)
defined function borrarCarpeta
carpetaInput: String = "C:/Curso-Scala/stream-input"
carpetaCheck: String = "C:/Curso-Scala/stream-checkpoint"
datosT

In [2]:
// CELDA T1 — Crear un stream y verificar sus propiedades básicas
// ⚠️ schema OBLIGATORIO en streaming — no existe inferSchema
val streamTeo = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")

println(s"¿Es streaming?  : ${streamTeo.isStreaming}")
println(s"¿Es batch?      : ${!streamTeo.isStreaming}")
println(s"\nSchema del stream:")
streamTeo.printSchema()
println("El stream está listo. No ha procesado datos aún — solo ha declarado la fuente.")

¿Es streaming?  : true
¿Es batch?      : false

Schema del stream:
root
 |-- timestamp: string (nullable = true)
 |-- usuario: string (nullable = true)
 |-- accion: string (nullable = true)
 |-- ciudad: string (nullable = true)

El stream está listo. No ha procesado datos aún — solo ha declarado la fuente.


streamTeo: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 2 more fields]

In [3]:
//filtrado y enriquecimiento
// CELDA T2 — Demostrar transformaciones: filter + withColumn
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val streamFiltrado = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .filter(col("accion") === "login")               // solo eventos de login
  .withColumn("procesado_en", current_timestamp()) // columna de tiempo de procesado

// Ejecutar y ver el resultado
val qT2 = streamFiltrado.writeStream
  .outputMode("append")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()                           // ← lanza el stream en background

qT2.awaitTermination()  // ← "espera aquí hasta que el stream pare"

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------+------+-------+-----------------------+
|timestamp          |usuario|accion|ciudad |procesado_en           |
+-------------------+-------+------+-------+-----------------------+
|2024-05-01 10:00:01|user001|login |Madrid |2026-05-06 12:12:42.091|
|2024-05-01 10:00:07|user003|login |Sevilla|2026-05-06 12:12:42.091|
+-------------------+-------+------+-------+-----------------------+



res3_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
streamFiltrado: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 3 more fields]
qT2: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@d6c02e8

In [4]:
// CELDA T3 — Demostrar agregación: groupBy + count
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val conteoAcciones = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .groupBy("accion")
  .count()
  .orderBy(col("count").desc)

// Con groupBy el outputMode debe ser "complete"
val qT3 = conteoAcciones.writeStream
  .outputMode("complete")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

qT3.awaitTermination()


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-----+
|accion  |count|
+--------+-----+
|login   |2    |
|pago    |1    |
|compra  |1    |
|busqueda|1    |
+--------+-----+



res4_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
conteoAcciones: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [accion: string, count: bigint]
qT3: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@7297c123

El siguiente ejemplo muestra el sink memory, que guarda los resultados en una tabla temporal en memoria que puedes consultar con SQL desde el mismo notebook:

In [5]:
// CELDA T4 — Sink memory: guardar resultado en tabla consultable con SQL
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val streamMem = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .groupBy("ciudad")
  .count()
  .orderBy(col("count").desc)

val qT4 = streamMem.writeStream
  .outputMode("complete")
  .format("memory")              // sink memory — guarda en tabla temporal
  .queryName("conteo_ciudades")  // nombre de la tabla SQL
  .trigger(Trigger.AvailableNow())
  .start()

qT4.awaitTermination()

// Consultar la tabla con Spark SQL
println("=== Resultado en tabla memory 'conteo_ciudades' ===")
spark.sql("SELECT * FROM conteo_ciudades").show()

=== Resultado en tabla memory 'conteo_ciudades' ===
+---------+-----+
|   ciudad|count|
+---------+-----+
|Barcelona|    2|
|   Madrid|    2|
|  Sevilla|    1|
+---------+-----+



res5_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
streamMem: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, count: bigint]
qT4: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@5c76aa10

In [6]:
//append — solo las filas nuevas de cada batch

// CELDA T5 — Demostrar output mode "append"
// Preparar dos lotes para ver la diferencia entre batches
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
borrarCarpeta("C:/Curso-Scala/stream-input")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-input"))

// Solo lote 1 por ahora
val lote1 =
  """timestamp,usuario,accion,ciudad
    |2024-05-01 10:00:01,user001,login,Madrid
    |2024-05-01 10:00:03,user002,compra,Barcelona
    |2024-05-01 10:00:05,user001,busqueda,Madrid""".stripMargin

Files.write(
  Paths.get("C:/Curso-Scala/stream-input/lote1.csv"),
  lote1.getBytes(StandardCharsets.UTF_8)
)

val qAppend = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .select("usuario", "accion", "ciudad")
  .writeStream
  .outputMode("append")          // solo escribe las filas NUEVAS de este batch
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

qAppend.awaitTermination()
println("→ append escribe SOLO las 3 filas nuevas del batch. No acumula.")

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+--------+---------+
|usuario|accion  |ciudad   |
+-------+--------+---------+
|user001|login   |Madrid   |
|user002|compra  |Barcelona|
|user001|busqueda|Madrid   |
+-------+--------+---------+

→ append escribe SOLO las 3 filas nuevas del batch. No acumula.


res6_2: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
res6_3: java.nio.file.Path = C:\Curso-Scala\stream-input
lote1: String = """timestamp,usuario,accion,ciudad
2024-05-01 10:00:01,user001,login,Madrid
2024-05-01 10:00:03,user002,compra,Barcelona
2024-05-01 10:00:05,user001,busqueda,Madrid"""
res6_5: java.nio.file.Path = C:\Curso-Scala\stream-input\lote1.csv
qAppend: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@2312be7f

In [7]:
//complete — el resultado acumulado completo en cada batch

// CELDA T6 — Demostrar output mode "complete"
// Restaurar el fichero de teoría con 5 eventos
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
borrarCarpeta("C:/Curso-Scala/stream-input")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-input"))

val datos5 =
  """timestamp,usuario,accion,ciudad
    |2024-05-01 10:00:01,user001,login,Madrid
    |2024-05-01 10:00:03,user002,compra,Barcelona
    |2024-05-01 10:00:05,user001,busqueda,Madrid
    |2024-05-01 10:00:07,user003,login,Sevilla
    |2024-05-01 10:00:09,user002,pago,Barcelona""".stripMargin

Files.write(
  Paths.get("C:/Curso-Scala/stream-input/eventos_teoria.csv"),
  datos5.getBytes(StandardCharsets.UTF_8)
)

val qComplete = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .groupBy("ciudad")
  .count()
  .orderBy(col("count").desc)
  .writeStream
  .outputMode("complete")        // reescribe el conteo COMPLETO en cada batch
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

qComplete.awaitTermination()
println("→ complete reescribe TODO el resultado acumulado. Necesario con groupBy.")

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+
|ciudad   |count|
+---------+-----+
|Barcelona|2    |
|Madrid   |2    |
|Sevilla  |1    |
+---------+-----+

→ complete reescribe TODO el resultado acumulado. Necesario con groupBy.


res7_2: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
res7_3: java.nio.file.Path = C:\Curso-Scala\stream-input
datos5: String = """timestamp,usuario,accion,ciudad
2024-05-01 10:00:01,user001,login,Madrid
2024-05-01 10:00:03,user002,compra,Barcelona
2024-05-01 10:00:05,user001,busqueda,Madrid
2024-05-01 10:00:07,user003,login,Sevilla
2024-05-01 10:00:09,user002,pago,Barcelona"""
res7_5: java.nio.file.Path = C:\Curso-Scala\stream-input\eventos_teoria.csv
qComplete: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@7d494018

In [8]:
// CELDA T7 — Estructura completa de un job de streaming
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

// 1. Schema (obligatorio)
// schemaEventos ya está definido en Celda T0

// 2. Source
val streamCompleto = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")

// 3. Transformación
val resultado = streamCompleto
  .filter(col("ciudad") =!= "Sevilla")   // excluir Sevilla
  .withColumn("es_compra", col("accion") === "compra")
  .groupBy("ciudad")
  .agg(
    count("*").alias("eventos"),
    sum(col("es_compra").cast("int")).alias("compras")
  )
  .orderBy(col("eventos").desc)

// 4. Sink + trigger + arrancar
val queryCompleto = resultado.writeStream
  .outputMode("complete")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

// 5. Esperar a que termine
queryCompleto.awaitTermination()

// 6. Inspeccionar progreso
println(s"\n=== Resumen de ejecución ===")
println(s"  Batches procesados : ${queryCompleto.lastProgress.batchId + 1}")
println(s"  Filas de entrada   : ${queryCompleto.lastProgress.numInputRows}")

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-------+-------+
|ciudad   |eventos|compras|
+---------+-------+-------+
|Barcelona|2      |1      |
|Madrid   |2      |0      |
+---------+-------+-------+


=== Resumen de ejecución ===
  Batches procesados : 1
  Filas de entrada   : 4


res8_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
streamCompleto: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 2 more fields]
resultado: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, eventos: bigint ... 1 more field]
queryCompleto: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@714cdf10

En producción real
En un sistema real (por ejemplo, leyendo de Kafka), el checkpoint guarda los offsets — básicamente el número de mensaje hasta el que se ha leído en cada partición. Si el servidor cae a las 3am y se reinicia a las 3:05, el stream no pierde ningún evento y no duplica ninguno. Eso es lo que hace al Structured Streaming fiable para sistemas críticos.

In [9]:
// CELDA T8 — Ver el efecto del checkpoint: no reprocesar ficheros ya vistos
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

// Primera ejecución — procesa eventos_teoria.csv
val qCheck1 = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .writeStream
  .outputMode("append")
  .format("console")
  .option("truncate", "false")
  .option("checkpointLocation", "C:/Curso-Scala/stream-checkpoint")
  .trigger(Trigger.AvailableNow())
  .start()

qCheck1.awaitTermination()
println(s"1ª ejecución — filas procesadas: ${qCheck1.lastProgress.numInputRows}")

// Segunda ejecución con el MISMO checkpoint — Spark recuerda que ya leyó ese fichero
val qCheck2 = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .writeStream
  .outputMode("append")
  .format("console")
  .option("truncate", "false")
  .option("checkpointLocation", "C:/Curso-Scala/stream-checkpoint")
  .trigger(Trigger.AvailableNow())
  .start()

qCheck2.awaitTermination()
println(s"2ª ejecución — filas procesadas: ${qCheck2.lastProgress.numInputRows}")
println("→ 0 filas: el checkpoint evitó reprocesar el mismo fichero.")

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------+--------+---------+
|timestamp          |usuario|accion  |ciudad   |
+-------------------+-------+--------+---------+
|2024-05-01 10:00:01|user001|login   |Madrid   |
|2024-05-01 10:00:03|user002|compra  |Barcelona|
|2024-05-01 10:00:05|user001|busqueda|Madrid   |
|2024-05-01 10:00:07|user003|login   |Sevilla  |
|2024-05-01 10:00:09|user002|pago    |Barcelona|
+-------------------+-------+--------+---------+

1ª ejecución — filas procesadas: 5
2ª ejecución — filas procesadas: 0
→ 0 filas: el checkpoint evitó reprocesar el mismo fichero.


res9_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
qCheck1: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@3446cb9a
qCheck2: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@45d00b5c

Parte 2 — Práctica

 Celda P0 — Preparar datos frescos para la práctica

In [10]:
// CELDA P0 — Limpiar estado de la teoría y preparar datos de práctica
val carpetaInput  = "C:/Curso-Scala/stream-input"
val carpetaOutput = "C:/Curso-Scala/stream-output"
val carpetaCheck  = "C:/Curso-Scala/stream-checkpoint"

Seq(carpetaInput, carpetaOutput, carpetaCheck).foreach { c =>
  borrarCarpeta(c)
  Files.createDirectories(Paths.get(c))
}

val lote1 =
  """timestamp,usuario,accion,ciudad
    |2024-05-01 10:00:01,user001,login,Madrid
    |2024-05-01 10:00:03,user002,compra,Barcelona
    |2024-05-01 10:00:05,user001,busqueda,Madrid
    |2024-05-01 10:00:07,user003,login,Sevilla
    |2024-05-01 10:00:09,user002,pago,Barcelona""".stripMargin

Files.write(
  Paths.get(s"$carpetaInput/eventos_lote1.csv"),
  lote1.getBytes(StandardCharsets.UTF_8)
)

println("✅ Entorno de práctica listo")
println(s"✅ Lote 1 escrito: eventos_lote1.csv (5 eventos)")

✅ Entorno de práctica listo
✅ Lote 1 escrito: eventos_lote1.csv (5 eventos)


carpetaInput: String = "C:/Curso-Scala/stream-input"
carpetaOutput: String = "C:/Curso-Scala/stream-output"
carpetaCheck: String = "C:/Curso-Scala/stream-checkpoint"
lote1: String = """timestamp,usuario,accion,ciudad
2024-05-01 10:00:01,user001,login,Madrid
2024-05-01 10:00:03,user002,compra,Barcelona
2024-05-01 10:00:05,user001,busqueda,Madrid
2024-05-01 10:00:07,user003,login,Sevilla
2024-05-01 10:00:09,user002,pago,Barcelona"""
res10_5: java.nio.file.Path = C:\Curso-Scala\stream-input\eventos_lote1.csv

In [11]:
//P1 — Primer stream: leer ficheros y mostrar en consola
// CELDA P1 — Stream básico: leer CSV → mostrar en consola
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val streamDF = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")

println(s"¿Es streaming?: ${streamDF.isStreaming}")
println("Schema del stream:")
streamDF.printSchema()

val query1 = streamDF.writeStream
  .outputMode("append")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

query1.awaitTermination()
println("✅ Stream P1 completado")

¿Es streaming?: true
Schema del stream:
root
 |-- timestamp: string (nullable = true)
 |-- usuario: string (nullable = true)
 |-- accion: string (nullable = true)
 |-- ciudad: string (nullable = true)

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------+--------+---------+
|timestamp          |usuario|accion  |ciudad   |
+-------------------+-------+--------+---------+
|2024-05-01 10:00:01|user001|login   |Madrid   |
|2024-05-01 10:00:03|user002|compra  |Barcelona|
|2024-05-01 10:00:05|user001|busqueda|Madrid   |
|2024-05-01 10:00:07|user003|login   |Sevilla  |
|2024-05-01 10:00:09|user002|pago    |Barcelona|
+-------------------+-------+--------+---------+

✅ Stream P1 completado


res11_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
streamDF: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 2 more fields]
query1: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@391bb567

In [12]:
//P2 — Stream con filtro: solo eventos de login
// CELDA P2 — Filtrar eventos en tiempo real
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val streamFiltrado = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .filter(col("accion") === "login")
  .withColumn("procesado_en", current_timestamp())

val query2 = streamFiltrado.writeStream
  .outputMode("append")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

query2.awaitTermination()
println("✅ Stream con filtro completado")

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------+------+-------+-----------------------+
|timestamp          |usuario|accion|ciudad |procesado_en           |
+-------------------+-------+------+-------+-----------------------+
|2024-05-01 10:00:01|user001|login |Madrid |2026-05-06 12:12:58.649|
|2024-05-01 10:00:07|user003|login |Sevilla|2026-05-06 12:12:58.649|
+-------------------+-------+------+-------+-----------------------+

✅ Stream con filtro completado


res12_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
streamFiltrado: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 3 more fields]
query2: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@18931820

In [13]:
// P3 — Stream con agregación: contar eventos por acción

// CELDA P3 — Agregación sobre un stream (output mode: complete)
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val conteoAcciones = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .groupBy("accion")
  .count()
  .orderBy(col("count").desc)

// ⚠️ Con groupBy el outputMode debe ser "complete"
val query3 = conteoAcciones.writeStream
  .outputMode("complete")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

query3.awaitTermination()
println("✅ Stream con agregación completado")

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-----+
|accion  |count|
+--------+-----+
|login   |2    |
|pago    |1    |
|compra  |1    |
|busqueda|1    |
+--------+-----+

✅ Stream con agregación completado


res13_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
conteoAcciones: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [accion: string, count: bigint]
query3: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@46996c6b

In [14]:
//P4 — Añadir un segundo lote de datos

// CELDA P4 — Escribir el segundo lote en la carpeta de entrada
val lote2 =
  """timestamp,usuario,accion,ciudad
    |2024-05-01 10:00:11,user004,login,Madrid
    |2024-05-01 10:00:13,user001,compra,Madrid
    |2024-05-01 10:00:15,user003,busqueda,Sevilla
    |2024-05-01 10:00:17,user005,login,Valencia
    |2024-05-01 10:00:19,user002,busqueda,Barcelona""".stripMargin

Files.write(
  Paths.get("C:/Curso-Scala/stream-input/eventos_lote2.csv"),
  lote2.getBytes(StandardCharsets.UTF_8)
)

println("✅ Lote 2 escrito: eventos_lote2.csv (5 eventos nuevos)")
println("   La carpeta stream-input ahora tiene 2 ficheros CSV (10 eventos en total)")

✅ Lote 2 escrito: eventos_lote2.csv (5 eventos nuevos)
   La carpeta stream-input ahora tiene 2 ficheros CSV (10 eventos en total)


lote2: String = """timestamp,usuario,accion,ciudad
2024-05-01 10:00:11,user004,login,Madrid
2024-05-01 10:00:13,user001,compra,Madrid
2024-05-01 10:00:15,user003,busqueda,Sevilla
2024-05-01 10:00:17,user005,login,Valencia
2024-05-01 10:00:19,user002,busqueda,Barcelona"""
res14_1: java.nio.file.Path = C:\Curso-Scala\stream-input\eventos_lote2.csv

In [15]:
//P5 — Relanzar el stream con los dos lotes
// CELDA P5 — Con checkpoint limpio, Spark procesa todos los ficheros disponibles
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val conteoTotal = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .groupBy("accion")
  .count()
  .orderBy(col("count").desc)

val query5 = conteoTotal.writeStream
  .outputMode("complete")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

query5.awaitTermination()
println("✅ Stream con 10 eventos (2 lotes) completado")

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-----+
|accion  |count|
+--------+-----+
|login   |4    |
|busqueda|3    |
|compra  |2    |
|pago    |1    |
+--------+-----+

✅ Stream con 10 eventos (2 lotes) completado


res15_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
conteoTotal: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [accion: string, count: bigint]
query5: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@24cb9b30

In [16]:
//P6 — Escribir resultados en disco (sink Parquet)
// CELDA P6 — Guardar el stream procesado en Parquet
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val streamEnriquecido = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")
  .withColumn("procesado_en", current_timestamp())
  .withColumn("es_login", col("accion") === "login")

val query6 = streamEnriquecido.writeStream
  .outputMode("append")
  .format("parquet")
  .option("checkpointLocation", "C:/Curso-Scala/stream-checkpoint")
  .trigger(Trigger.AvailableNow())
  .start("C:/Curso-Scala/stream-output/eventos_procesados")

query6.awaitTermination()
println("✅ Stream escrito en Parquet")

val dfResultado = spark.read
  .parquet("C:/Curso-Scala/stream-output/eventos_procesados")
println(s"\n=== Resultado guardado: ${dfResultado.count()} filas ===")
dfResultado.show(truncate = false)

✅ Stream escrito en Parquet

=== Resultado guardado: 10 filas ===
+-------------------+-------+--------+---------+-----------------------+--------+
|timestamp          |usuario|accion  |ciudad   |procesado_en           |es_login|
+-------------------+-------+--------+---------+-----------------------+--------+
|2024-05-01 10:00:01|user001|login   |Madrid   |2026-05-06 12:13:03.955|true    |
|2024-05-01 10:00:03|user002|compra  |Barcelona|2026-05-06 12:13:03.955|false   |
|2024-05-01 10:00:05|user001|busqueda|Madrid   |2026-05-06 12:13:03.955|false   |
|2024-05-01 10:00:07|user003|login   |Sevilla  |2026-05-06 12:13:03.955|true    |
|2024-05-01 10:00:09|user002|pago    |Barcelona|2026-05-06 12:13:03.955|false   |
|2024-05-01 10:00:11|user004|login   |Madrid   |2026-05-06 12:13:03.955|true    |
|2024-05-01 10:00:13|user001|compra  |Madrid   |2026-05-06 12:13:03.955|false   |
|2024-05-01 10:00:15|user003|busqueda|Sevilla  |2026-05-06 12:13:03.955|false   |
|2024-05-01 10:00:17|user005|log

res16_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
streamEnriquecido: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 4 more fields]
query6: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@58a11326
dfResultado: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 4 more fields]

In [17]:
//P7 — Monitorizar el progreso de un stream
// CELDA P7 — Inspeccionar el estado y progreso tras completar
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

val streamMonitor = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")

val queryMonitor = streamMonitor.writeStream
  .outputMode("append")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

queryMonitor.awaitTermination()

val progreso = queryMonitor.lastProgress
println("\n=== Estado del stream ===")
println(s"  ID único         : ${queryMonitor.id}")
println(s"  Último batch ID  : ${progreso.batchId}")
println(s"  Filas procesadas : ${progreso.numInputRows}")
println(s"  Estado           : ${if (queryMonitor.isActive) "activo" else "completado"}")

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------+--------+---------+
|timestamp          |usuario|accion  |ciudad   |
+-------------------+-------+--------+---------+
|2024-05-01 10:00:11|user004|login   |Madrid   |
|2024-05-01 10:00:13|user001|compra  |Madrid   |
|2024-05-01 10:00:15|user003|busqueda|Sevilla  |
|2024-05-01 10:00:17|user005|login   |Valencia |
|2024-05-01 10:00:19|user002|busqueda|Barcelona|
|2024-05-01 10:00:01|user001|login   |Madrid   |
|2024-05-01 10:00:03|user002|compra  |Barcelona|
|2024-05-01 10:00:05|user001|busqueda|Madrid   |
|2024-05-01 10:00:07|user003|login   |Sevilla  |
|2024-05-01 10:00:09|user002|pago    |Barcelona|
+-------------------+-------+--------+---------+


=== Estado del stream ===
  ID único         : 57f315db-5473-4687-888c-a7c0afbda7db
  Último batch ID  : 0
  Filas procesadas : 10
  Estado           : completado


res17_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
streamMonitor: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 2 more fields]
queryMonitor: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@154717b9
progreso: org.apache.spark.sql.streaming.StreamingQueryProgress = {
  "id" : "57f315db-5473-4687-888c-a7c0afbda7db",
  "runId" : "56e9eafb-2e2e-4f99-ac67-18d9185f7dd6",
  "name" : null,
  "timestamp" : "2026-05-06T10:13:06.177Z",
  "batchId" : 0,
  "batchDuration" : 562,
  "numInputRows" : 10,
  "inputRowsPerSecond" : 0.0,
  "processedRowsPerSecond" : 17.8,
  "durationMs" : {
    "addBatch" : 47,
    "commitOffsets" : 165,
    "getBatch" : 5,
    "latestOffset" : 163,
    "queryPlanning" : 4,
    "triggerExecution" : 561,
    "walCommit" : 176
  },
...

In [18]:
//Reto final — Pipeline de análisis de eventos en tiempo real

// CELDA RETO — Pipeline completo: ingestar, transformar, agregar y mostrar
borrarCarpeta("C:/Curso-Scala/stream-checkpoint")
Files.createDirectories(Paths.get("C:/Curso-Scala/stream-checkpoint"))

// Paso 1: Leer el stream
val rawStream = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv("C:/Curso-Scala/stream-input/")

// Paso 2: Clasificar si el evento es una conversión (compra o pago)
val streamTransformado = rawStream
  .withColumn("procesado_en",  current_timestamp())
  .withColumn("es_conversion", col("accion").isin("compra", "pago"))

// Paso 3: Agregar — resumen por ciudad
val resumenCiudades = streamTransformado
  .groupBy("ciudad")
  .agg(
    count("*").alias("total_eventos"),
    sum(col("es_conversion").cast("int")).alias("conversiones")
  )
  .orderBy(col("total_eventos").desc)

// Paso 4: Mostrar en consola
val queryReto = resumenCiudades.writeStream
  .outputMode("complete")
  .format("console")
  .option("truncate", "false")
  .trigger(Trigger.AvailableNow())
  .start()

queryReto.awaitTermination()

println("\n✅ Pipeline completado")
println(s"   Eventos procesados: ${queryReto.lastProgress.numInputRows}")

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-------------+------------+
|ciudad   |total_eventos|conversiones|
+---------+-------------+------------+
|Madrid   |4            |1           |
|Barcelona|3            |2           |
|Sevilla  |2            |0           |
|Valencia |1            |0           |
+---------+-------------+------------+


✅ Pipeline completado
   Eventos procesados: 10


res18_1: java.nio.file.Path = C:\Curso-Scala\stream-checkpoint
rawStream: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 2 more fields]
streamTransformado: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 4 more fields]
resumenCiudades: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, total_eventos: bigint ... 1 more field]
queryReto: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@6e5ab04a

In [18]:
//Hacer pruebas de streaming processing en tu spark-shell.
 //Usando el output mode : console y cambiando también prueba cambiando el
  //Trigger AvailableNow() por ProcessingTime() . Adjunta algunas capturas en tu entrega



In [19]:
val pathInput = "C:/Curso-Scala/stream-input/"


pathInput: String = "C:/Curso-Scala/stream-input/"

In [20]:
//con Trigger.AvailableNow()
val streamBatch = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv(pathInput)

val query1 = streamBatch.writeStream
  .outputMode("append")
  .format("console")
  .trigger(Trigger.AvailableNow())
  .start()

query1.awaitTermination() 
// Verás el resultado en consola y el prompt (scala>) volverá a aparecer.


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------+--------+---------+
|          timestamp|usuario|  accion|   ciudad|
+-------------------+-------+--------+---------+
|2024-05-01 10:00:11|user004|   login|   Madrid|
|2024-05-01 10:00:13|user001|  compra|   Madrid|
|2024-05-01 10:00:15|user003|busqueda|  Sevilla|
|2024-05-01 10:00:17|user005|   login| Valencia|
|2024-05-01 10:00:19|user002|busqueda|Barcelona|
|2024-05-01 10:00:01|user001|   login|   Madrid|
|2024-05-01 10:00:03|user002|  compra|Barcelona|
|2024-05-01 10:00:05|user001|busqueda|   Madrid|
|2024-05-01 10:00:07|user003|   login|  Sevilla|
|2024-05-01 10:00:09|user002|    pago|Barcelona|
+-------------------+-------+--------+---------+



streamBatch: org.apache.spark.sql.package.DataFrame = [timestamp: string, usuario: string ... 2 more fields]
query1: org.apache.spark.sql.streaming.StreamingQuery = org.apache.spark.sql.execution.streaming.runtime.StreamingQueryWrapper@2de69313

In [21]:
//con Trigger.ProcessingTime("5 seconds")
val streamContinuo = spark.readStream
  .option("header", "true")
  .schema(schemaEventos)
  .csv(pathInput)

val query2 = streamContinuo.writeStream
  .outputMode("append")
  .format("console")
  .trigger(Trigger.ProcessingTime("5 seconds"))
  .start()

// ¡OJO! Esto bloqueará tu terminal
query2.awaitTermination() 


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------+--------+---------+
|          timestamp|usuario|  accion|   ciudad|
+-------------------+-------+--------+---------+
|2024-05-01 10:00:11|user004|   login|   Madrid|
|2024-05-01 10:00:13|user001|  compra|   Madrid|
|2024-05-01 10:00:15|user003|busqueda|  Sevilla|
|2024-05-01 10:00:17|user005|   login| Valencia|
|2024-05-01 10:00:19|user002|busqueda|Barcelona|
|2024-05-01 10:00:01|user001|   login|   Madrid|
|2024-05-01 10:00:03|user002|  compra|Barcelona|
|2024-05-01 10:00:05|user001|busqueda|   Madrid|
|2024-05-01 10:00:07|user003|   login|  Sevilla|
|2024-05-01 10:00:09|user002|    pago|Barcelona|
+-------------------+-------+--------+---------+



: 